### Basic neural activity analysis with single camera tracking
#### analyze the firing rate PC1,2,3
#### making the demo videos
#### analyze the spike triggered pull and gaze
#### analyze the bhv triggered firing rate and firing rate PC1,2,3
#### newly added!! use IRL on the bhv variables to infer the values
#### (not working very well so pause it)

In [ ]:
import pandas as pd
import numpy as np
from numpy import genfromtxt
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn
import scipy
import scipy.stats as st
import scipy.io
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from dPCA import dPCA
import string
import warnings
import pickle
import json

from scipy.ndimage import gaussian_filter1d

import os
import glob
import random
from time import time

from pgmpy.models import BayesianModel
from pgmpy.models import DynamicBayesianNetwork as DBN
from pgmpy.estimators import BayesianEstimator
from pgmpy.estimators import HillClimbSearch,BicScore
from pgmpy.base import DAG
import networkx as nx

from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests


### function - get body part location for each pair of cameras

In [ ]:
from ana_functions.body_part_locs_eachpair import body_part_locs_eachpair
from ana_functions.body_part_locs_singlecam import body_part_locs_singlecam

### function - align the two cameras

In [ ]:
from ana_functions.camera_align import camera_align       

### function - merge the two pairs of cameras

In [ ]:
from ana_functions.camera_merge import camera_merge

### function - find social gaze time point

In [ ]:
from ana_functions.find_socialgaze_timepoint import find_socialgaze_timepoint
from ana_functions.find_socialgaze_timepoint_singlecam import find_socialgaze_timepoint_singlecam
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody import find_socialgaze_timepoint_singlecam_wholebody

### function - define time point of behavioral events

In [ ]:
from ana_functions.bhv_events_timepoint import bhv_events_timepoint
from ana_functions.bhv_events_timepoint_singlecam import bhv_events_timepoint_singlecam

### function - plot behavioral events

In [ ]:
from ana_functions.plot_bhv_events import plot_bhv_events
from ana_functions.plot_bhv_events_levertube import plot_bhv_events_levertube
from ana_functions.draw_self_loop import draw_self_loop
import matplotlib.patches as mpatches 
from matplotlib.collections import PatchCollection

### function - plot inter-pull interval

In [ ]:
from ana_functions.plot_interpull_interval import plot_interpull_interval

### function - get the continuous behavioral variables

In [ ]:
from ana_functions.plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection import plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection
from ana_functions.plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection_highbhvDimension_to_lowPCspace import plot_continuous_bhv_var_singlecam_PullStartToPull_variedSection_highbhvDimension_to_lowPCspace
from ana_functions.singlecam_conBhv_from_highDimension_to_PCspace import get_data_for_singlecam_conBhv_from_highDimension_to_PCspace

### function - make demo videos with skeleton and inportant vectors

In [ ]:
from ana_functions.tracking_video_singlecam_demo import tracking_video_singlecam_demo
from ana_functions.tracking_video_singlecam_wholebody_demo import tracking_video_singlecam_wholebody_demo
from ana_functions.tracking_video_singlecam_wholebody_withNeuron_demo import tracking_video_singlecam_wholebody_withNeuron_demo
from ana_functions.tracking_video_singlecam_wholebody_withNeuron_sepbhv_demo import tracking_video_singlecam_wholebody_withNeuron_sepbhv_demo
from ana_functions.tracking_frame_singlecam_wholebody_withNeuron_sepbhv_demo import tracking_frame_singlecam_wholebody_withNeuron_sepbhv_demo

### function - interval between all behavioral events

In [ ]:
from ana_functions.bhv_events_interval import bhv_events_interval

### function - spike analysis

In [ ]:
from ana_functions.spike_analysis_FR_calculation import spike_analysis_FR_calculation
from ana_functions.plot_spike_triggered_singlecam_bhvevent import plot_spike_triggered_singlecam_bhvevent
from ana_functions.plot_bhv_events_aligned_FR import plot_bhv_events_aligned_FR
from ana_functions.plot_strategy_aligned_FR import plot_strategy_aligned_FR

### function - PCA projection

In [ ]:
from ana_functions.PCA_around_bhv_events import PCA_around_bhv_events
from ana_functions.PCA_around_bhv_events_video import PCA_around_bhv_events_video
from ana_functions.confidence_ellipse import confidence_ellipse

### function - other useful functions; related to bhv glm

In [ ]:
from functions.continuous_variable_glm import continuous_variable_glm
from functions.continuous_variable_glm_shortlist_prediction import continuous_variable_glm_shortlist_prediction
from functions.continuous_variable_create_data_forGLM import continuous_variable_create_data_forGLM
from functions.continuous_variable_create_data_forGLM import fit_glm_and_predict
from functions.continuous_variable_create_data_forGLM import plot_glm_temporal_filters
from functions.continuous_variable_create_data_forGLM import plot_pull_likelihood

## Analyze each session

### prepare the basic behavioral data (especially the time stamps for each bhv events)

In [ ]:
# instead of using gaze angle threshold, use the target rectagon to deside gaze info
# ...need to update
sqr_thres_tubelever = 75 # draw the square around tube and lever
sqr_thres_face = 1.15 # a ratio for defining face boundary
sqr_thres_body = 4 # how many times to enlongate the face box boundry to the body


# get the fps of the analyzed video
fps = 30

# get the fs for neural recording
fs_spikes = 20000
fs_lfp = 1000

# frame number of the demo video
nframes = 0.5*30 # second*30fps
# nframes = 45*30 # second*30fps

# re-analyze the video or not
reanalyze_video = 0
redo_anystep = 1

# do OFC sessions or DLPFC sessions
do_OFC = 1
do_DLPFC  = 0
if do_OFC:
    savefile_sufix = '_OFCs'
elif do_DLPFC:
    savefile_sufix = '_DLPFCs'
else:
    savefile_sufix = ''
    
# all the videos (no misaligned ones)
# aligned with the audio
# get the session start time from "videosound_bhv_sync.py/.ipynb"
# currently the session_start_time will be manually typed in. It can be updated after a better method is used


# dodson ginger for dlpfc (dmpfc)
# dodson selene for ofc
if 1:
    if do_DLPFC:
        neural_record_conditions = [
                        '20240531_Dodson_MC', '20240603_Dodson_MC_and_SR', '20240603_Dodson_MC_and_SR',
                        '20240604_Dodson_MC', '20240605_Dodson_MC_and_SR', '20240605_Dodson_MC_and_SR',

                        '20240606_Dodson_MC_and_SR', '20240606_Dodson_MC_and_SR', '20240607_Dodson_SR',
                        '20240610_Dodson_MC', '20240611_Dodson_SR', '20240612_Dodson_MC',

                        '20240613_Dodson_SR', '20240620_Dodson_SR', '20240719_Dodson_MC',
                        '20250129_Dodson_MC', '20250130_Dodson_SR', '20250131_Dodson_MC',

                        '20250210_Dodson_SR_withKoala', '20250211_Dodson_MC_withKoala',
                        '20250212_Dodson_SR_withKoala', '20250214_Dodson_MC_withKoala',
                        '20250217_Dodson_SR_withKoala', '20250218_Dodson_MC_withKoala',

                        '20250219_Dodson_SR_withKoala', '20250220_Dodson_MC_withKoala',
                        '20250224_Dodson_KoalaAL_withKoala', '20250226_Dodson_MC_withKoala',
                        '20250227_Dodson_KoalaAL_withKoala', '20250228_Dodson_DodsonAL_withKoala',

                        '20250304_Dodson_DodsonAL_withKoala', '20250305_Dodson_MC_withKoala',
                        '20250306_Dodson_KoalaAL_withKoala', '20250307_Dodson_DodsonAL_withKoala',
                        '20250310_Dodson_MC_withKoala', '20250312_Dodson_NV_withKoala',

                        '20250313_Dodson_NV_withKoala', '20250314_Dodson_NV_withKoala',
                        '20250401_Dodson_MC_withKanga', '20250402_Dodson_MC_withKanga',
                        '20250403_Dodson_MC_withKanga', '20250404_Dodson_SR_withKanga',

                        '20250407_Dodson_SR_withKanga', '20250408_Dodson_SR_withKanga',
                        '20250409_Dodson_MC_withKanga', '20250415_Dodson_MC_withKanga',
                        # '20250416_Dodson_SR_withKanga',
                        '20250417_Dodson_MC_withKanga',

                        '20250418_Dodson_SR_withKanga', '20250421_Dodson_SR_withKanga',
                        '20250422_Dodson_MC_withKanga', '20250422_Dodson_SR_withKanga',
                        '20250423_Dodson_MC_withKanga', '20250423_Dodson_SR_withKanga',

                        '20250424_Dodson_NV_withKanga', '20250424_Dodson_MC_withKanga',
                        '20250424_Dodson_SR_withKanga', '20250425_Dodson_NV_withKanga',
                        '20250425_Dodson_SR_withKanga', '20250428_Dodson_NV_withKanga',

                        '20250428_Dodson_MC_withKanga', '20250428_Dodson_SR_withKanga',
                        '20250429_Dodson_NV_withKanga', '20250429_Dodson_MC_withKanga',
                        '20250429_Dodson_SR_withKanga', '20250430_Dodson_NV_withKanga',

                        '20250430_Dodson_MC_withKanga', '20250430_Dodson_SR_withKanga',
                    ]
        task_conditions = [
                        'MC', 'MC', 'SR', 'MC', 'MC', 'SR',
                        'MC', 'SR', 'SR', 'MC', 'SR', 'MC',
                        'SR', 'SR', 'MC', 'MC_withGingerNew', 'SR_withGingerNew', 'MC_withGingerNew',

                        'SR_withKoala', 'MC_withKoala', 'SR_withKoala',
                        'MC_withKoala', 'SR_withKoala', 'MC_withKoala',

                        'SR_withKoala', 'MC_withKoala', 'MC_KoalaAuto_withKoala',
                        'MC_withKoala', 'MC_KoalaAuto_withKoala', 'MC_DodsonAuto_withKoala',

                        'MC_DodsonAuto_withKoala', 'MC_withKoala', 'MC_KoalaAuto_withKoala',
                        'MC_DodsonAuto_withKoala', 'MC_withKoala', 'NV_withKoala',

                        'NV_withKoala', 'NV_withKoala', 'MC_withKanga',
                        'MC_withKanga', 'MC_withKanga', 'SR_withKanga',

                        'SR_withKanga', 'SR_withKanga', 'MC_withKanga',
                        'MC_withKanga',
                        # 'SR_withKanga',
                        'MC_withKanga', 'SR_withKanga', 'SR_withKanga', 'MC_withKanga', 'SR_withKanga',

                        'MC_withKanga', 'SR_withKanga', 'NV_withKanga',
                        'MC_withKanga', 'SR_withKanga', 'NV_withKanga',

                        'SR_withKanga', 'NV_withKanga', 'MC_withKanga',
                        'SR_withKanga', 'NV_withKanga', 'MC_withKanga',

                        'SR_withKanga', 'NV_withKanga', 'MC_withKanga', 'SR_withKanga',
                    ]
        dates_list = [
                        '20240531', '20240603_MC', '20240603_SR', '20240604', '20240605_MC', '20240605_SR',
                        '20240606_MC', '20240606_SR', '20240607', '20240610_MC', '20240611', '20240612',

                        '20240613', '20240620', '20240719',
                        '20250129', '20250130', '20250131',

                        '20250210', '20250211', '20250212', '20250214', '20250217', '20250218',
                        '20250219', '20250220', '20250224', '20250226', '20250227', '20250228',

                        '20250304', '20250305', '20250306', '20250307', '20250310', '20250312',
                        '20250313', '20250314',

                        '20250401', '20250402', '20250403', '20250404', '20250407', '20250408',
                        '20250409',

                        '20250415',
                        # '20250416',
                        '20250417', '20250418', '20250421', '20250422', '20250422_SR',

                        '20250423', '20250423_SR', '20250424', '20250424_MC', '20250424_SR',
                        '20250425', '20250425_SR',

                        '20250428_NV', '20250428_MC', '20250428_SR',
                        '20250429_NV', '20250429_MC', '20250429_SR',

                        '20250430_NV', '20250430_MC', '20250430_SR',
                    ]
        videodates_list = [
                        '20240531', '20240603', '20240603', '20240604', '20240605', '20240605',
                        '20240606', '20240606', '20240607', '20240610_MC', '20240611', '20240612',

                        '20240613', '20240620', '20240719',
                        '20250129', '20250130', '20250131',

                        '20250210', '20250211', '20250212', '20250214', '20250217', '20250218',
                        '20250219', '20250220', '20250224', '20250226', '20250227', '20250228',

                        '20250304', '20250305', '20250306', '20250307', '20250310', '20250312',
                        '20250313', '20250314',

                        '20250401', '20250402', '20250403', '20250404', '20250407', '20250408',
                        '20250409',

                        '20250415',
                        # '20250416',
                        '20250417', '20250418', '20250421', '20250422', '20250422_SR',

                        '20250423', '20250423_SR', '20250424', '20250424_MC', '20250424_SR',
                        '20250425', '20250425_SR',

                        '20250428_NV', '20250428_MC', '20250428_SR',
                        '20250429_NV', '20250429_MC', '20250429_SR',

                        '20250430_NV', '20250430_MC', '20250430_SR',
                    ] # to deal with the sessions that MC and SR were in the same session
        session_start_times = [
                        0.00, 340, 340, 72.0, 60.1, 60.1,
                        82.2, 82.2, 35.8, 0.00, 29.2, 35.8,

                        62.5, 71.5, 54.4,
                        0.00, 0.00, 0.00,

                        0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00, 0.00, 0.00, 0.00, 0.00, 0.00,

                        0.00, 0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00, 0.00,

                        0.00, 0.00, 73.5, 0.00, 76.1, 81.5,
                        0.00,

                        363,
                        # 0.00,
                        79.0, 162.6, 231.9, 109, 0.00,

                        0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00, 93.0,

                        0.00, 0.00, 0.00,
                        0.00, 0.00, 0.00,

                        0.00, 274.4, 0.00,
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0', #'Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             
                              ]
        animal1_fixedorders = ['dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',# 'dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson',
                              ]
        recordedanimals = animal1_fixedorders 
        animal2_fixedorders = ['ginger','ginger','ginger','ginger','ginger','ginger','ginger','ginger','ginger',
                               'ginger','ginger','ginger','ginger','ginger','ginger','gingerNew','gingerNew','gingerNew',
                               'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala',
                               'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala', 'koala',
                               'koala', 'koala', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', # 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                              ]

        animal1_filenames = ["Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",# "Dodson",
                             'Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson','Dodson',
                             'Dodson','Dodson','Dodson','Dodson','Dodson',
                            ]
        animal2_filenames = ["Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger",
                             "Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger","Ginger",
                             "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala",
                             "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala", "Koala",
                             "Koala", "Koala", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga", # "Kanga",
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                            ]
        
    elif do_OFC:
        neural_record_conditions = [
                        '20260219_Dodson_OFC_33turns_SRwithSelene',  '20260303_Dodson_OFC_33turns_1sMCwithSelene',
                        '20260303_Dodson_OFC_33turns_SRwithSelene',  '20260304_Dodson_OFC_33turns_1sMCwithSelene',
                        '20260304_Dodson_OFC_33turns_SRwithSelene',  '20260305_Dodson_OFC_33turns_1sMCwithSelene',
                        '20260309_Dodson_OFC_33turns_1sMCwithKanga', '20260309_Dodson_OFC_33turns_SRwithKanga',
                        '20260310_Dodson_OFC_33turns_1sMCwithKanga', '20260310_Dodson_OFC_33turns_SRwithKanga',
                        '20260311_Dodson_OFC_33turns_1sMCwithKanga', '20260311_Dodson_OFC_33turns_SRwithKanga',
                        '20260312_Dodson_OFC_33turns_1sMCwithKanga', '20260312_Dodson_OFC_33turns_SRwithKanga',
                        '20260313_Dodson_OFC_33turns_1sMCwithKanga', '20260313_Dodson_OFC_33turns_SRwithKanga',
            
                        '20260317_Dodson_OFC_33turns_1sMCwithKanga', '20260317_Dodson_OFC_33turns_SRwithKanga',
                        '20260318_Dodson_OFC_33turns_1sMCwithKanga', # '20260318_Dodson_OFC_33turns_SRwithKanga',
                        '20260319_Dodson_OFC_33turns_1sMCwithKanga', '20260319_Dodson_OFC_33turns_SRwithKanga',
                        '20260323_Dodson_OFC_32turns_1sMCwithKanga', '20260323_Dodson_OFC_32turns_SRwithKanga',
                        '20260324_Dodson_OFC_32turns_1sMCwithKanga', '20260324_Dodson_OFC_32turns_SRwithKanga',
                    ]
        task_conditions = [
                        'SR', 'MC', 'SR', 'MC', 'SR', 'MC', 
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC',       'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 
                    ]
        dates_list = [
                        '20260219', '20260303',    '20260303_SR', '20260304',    '20260304_SR', '20260305', 
                        '20260309', '20260309_SR', '20260310',    '20260310_SR', '20260311',    '20260311_SR',
                        '20260312', '20260312_SR', '20260313',    '20260313_SR', '20260317',    '20260317_SR',
                        '20260318',                '20260319',    '20260319_SR', '20260323',    '20260323_SR',
                        '20260324', '20260324_SR',
                    ]
        videodates_list = [
                        '20260219', '20260303',    '20260303_SR', '20260304',    '20260304_SR', '20260305', 
                        '20260309', '20260309_SR', '20260310',    '20260310_SR', '20260311',    '20260311_SR',
                        '20260312', '20260312_SR', '20260313',    '20260313_SR', '20260317',    '20260317_SR',
                        '20260318',                '20260319',    '20260319_SR', '20260323',    '20260323_SR',
                        '20260324', '20260324_SR',
                    ] 
        
        session_start_times = [
                        188.7, 0.00, 0.00, 0.00, 0.00,  0.00, 
                         0.00, 0.00, 0.00, 0.00, 0.00,  0.00, 
                         0.00, 0.00, 0.00, 0.00, 0.00,  0.00, 
                         0.00,       0.00, 0.00, 0.00, 129.5,
                        116.2, 0.00,
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai9',           'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              'Dev1/ai9','Dev1/ai9',
                              ]
        animal1_fixedorders = ['dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson',         'dodson','dodson','dodson','dodson',
                               'dodson','dodson',
                              ]
        recordedanimals = animal1_fixedorders 
        animal2_fixedorders = ['selene','selene','selene','selene','selene','selene',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga',          'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga',
                              ]

        animal1_filenames = ["Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson",         "Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson",
                            ]
        animal2_filenames = ['Selene','Selene','Selene','Selene','Selene','Selene',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga',          'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga',
                            ]


    
# dannon kanga
if 1:
    if do_DLPFC:
        neural_record_conditions = [
                        '20240508_Kanga_SR', '20240509_Kanga_MC', '20240513_Kanga_MC',
                        '20240514_Kanga_SR', '20240523_Kanga_MC', '20240524_Kanga_SR',

                        '20240606_Kanga_MC', '20240613_Kanga_MC_DannonAuto',
                        '20240614_Kanga_MC_DannonAuto', '20240617_Kanga_MC_DannonAuto',
                        '20240618_Kanga_MC_KangaAuto', '20240619_Kanga_MC_KangaAuto',

                        '20240620_Kanga_MC_KangaAuto', '20240621_1_Kanga_NoVis',
                        '20240624_Kanga_NoVis', '20240626_Kanga_NoVis',

                        '20240808_Kanga_MC_withGinger', '20240809_Kanga_MC_withGinger',
                        '20240812_Kanga_MC_withGinger', '20240813_Kanga_MC_withKoala',
                        '20240814_Kanga_MC_withKoala', '20240815_Kanga_MC_withKoala',

                        '20240819_Kanga_MC_withVermelho', '20240821_Kanga_MC_withVermelho',
                        '20240822_Kanga_MC_withVermelho',

                        '20250415_Kanga_MC_withDodson', '20250416_Kanga_SR_withDodson',
                        '20250417_Kanga_MC_withDodson', '20250418_Kanga_SR_withDodson',
                        '20250421_Kanga_SR_withDodson', '20250422_Kanga_MC_withDodson',

                        '20250422_Kanga_SR_withDodson', '20250423_Kanga_MC_withDodson',
                        '20250423_Kanga_SR_withDodson',

                        '20250424_Kanga_NV_withDodson', '20250424_Kanga_MC_withDodson',
                        '20250424_Kanga_SR_withDodson', '20250425_Kanga_NV_withDodson',
                        '20250425_Kanga_SR_withDodson',

                        '20250428_Kanga_NV_withDodson', '20250428_Kanga_MC_withDodson',
                        '20250428_Kanga_SR_withDodson', '20250429_Kanga_NV_withDodson',
                        '20250429_Kanga_MC_withDodson', '20250429_Kanga_SR_withDodson',

                        '20250430_Kanga_NV_withDodson', '20250430_Kanga_MC_withDodson',
                        '20250430_Kanga_SR_withDodson',
                    ]
        dates_list = [
                        "20240508", "20240509", "20240513", "20240514", "20240523", "20240524",
                        "20240606", "20240613", "20240614", "20240617", "20240618", "20240619",
                        "20240620", "20240621_1", "20240624", "20240626",

                        "20240808", "20240809", "20240812", "20240813", "20240814", "20240815",
                        "20240819", "20240821", "20240822",

                        "20250415", "20250416", "20250417", "20250418", "20250421", "20250422",
                        "20250422_SR",

                        '20250423', '20250423_SR', '20250424', '20250424_MC', '20250424_SR',
                        '20250425', '20250425_SR',

                        '20250428_NV', '20250428_MC', '20250428_SR',
                        '20250429_NV', '20250429_MC', '20250429_SR',

                        '20250430_NV', '20250430_MC', '20250430_SR',
                    ]
        videodates_list = dates_list
        task_conditions = [
                        'SR', 'MC', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'MC_DannonAuto', 'MC_DannonAuto', 'MC_DannonAuto',
                        'MC_KangaAuto', 'MC_KangaAuto',

                        'MC_KangaAuto', 'NV', 'NV', 'NV',

                        'MC_withGinger', 'MC_withGinger', 'MC_withGinger',
                        'MC_withKoala', 'MC_withKoala', 'MC_withKoala',

                        'MC_withVermelho', 'MC_withVermelho', 'MC_withVermelho',

                        'MC_withDodson', 'SR_withDodson', 'MC_withDodson',
                        'SR_withDodson', 'SR_withDodson', 'MC_withDodson',

                        'SR_withDodson', 'MC_withDodson', 'SR_withDodson',

                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',
                        'NV_withDodson', 'SR_withDodson',

                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',
                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',

                        'NV_withDodson', 'MC_withDodson', 'SR_withDodson',
                    ]
        session_start_times = [
                        0.00, 36.0, 69.5, 0.00, 62.0, 0.00,
                        89.0, 0.00, 0.00, 0.00, 165.8, 96.0,
            
                        0.00, 0.00, 0.00, 48.0,
                        59.2, 49.5, 40.0, 50.0, 0.00, 69.8,
            
                        85.0, 212.9, 68.5,
                        363, 0.00, 79.0, 162.6, 231.9, 109,
            
                        0.00,
                        0.00, 0.00, 0.00, 0.00, 0.00,

                        0.00, 93.0,

                        0.00, 0.00, 0.00, 0.00, 0.00,
                        0.00,

                        0.00, 274.4, 0.00,
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai0','Dev1/ai0',
                             'Dev1/ai0','Dev1/ai0','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                             'Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9','Dev1/ai9',
                              ]
        
        animal1_fixedorders = ['dannon','dannon','dannon','dannon','dannon','dannon','dannon','dannon',
                               'dannon','dannon','dannon','dannon','dannon','dannon','dannon','dannon',
                               'ginger','ginger','ginger','koala','koala','koala','vermelho','vermelho',
                               'vermelho','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson','dodson','dodson',
                              ]
        animal2_fixedorders = ['kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                               'kanga','kanga','kanga','kanga','kanga','kanga','kanga','kanga',
                              ]
        recordedanimals = animal2_fixedorders

        animal1_filenames = ["Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon",
                             "Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon","Dannon",
                             "Ginger","Ginger","Ginger", "Kanga", "Kanga", "Kanga", "Kanga", "Kanga",
                              "Kanga","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             
                            ]
        animal2_filenames = ["Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Koala","Koala","Koala","Vermelho","Vermelho",
                             "Vermelho","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                             "Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga","Kanga",
                            ]
        
    elif do_OFC:
        neural_record_conditions = [
                        '20260309_Kanga_OFC_31turns_1sMCwithDodson', '20260309_Kanga_OFC_31turns_SRwithDodson',
                        '20260310_Kanga_OFC_31turns_1sMCwithDodson', '20260310_Kanga_OFC_31turns_SRwithDodson',
                        '20260311_Kanga_OFC_31turns_1sMCwithDodson', '20260311_Kanga_OFC_31turns_SRwithDodson',
                        '20260312_Kanga_OFC_31turns_1sMCwithDodson', '20260312_Kanga_OFC_31turns_SRwithDodson',
                        '20260313_Kanga_OFC_31turns_1sMCwithDodson', '20260313_Kanga_OFC_31turns_SRwithDodson',
                        '20260317_Kanga_OFC_31turns_1sMCwithDodson', '20260317_Kanga_OFC_31turns_SRwithDodson',
                        '20260318_Kanga_OFC_31turns_1sMCwithDodson', '20260318_Kanga_OFC_31turns_SRwithDodson',
                        '20260319_Kanga_OFC_31turns_1sMCwithDodson', '20260319_Kanga_OFC_31turns_SRwithDodson',
                        '20260323_Kanga_OFC_31turns_1sMCwithDodson', '20260323_Kanga_OFC_31turns_SRwithDodson',
                        '20260324_Kanga_OFC_31turns_1sMCwithDodson', '20260324_Kanga_OFC_31turns_SRwithDodson',
                        '20260326_Kanga_OFC_31turns_1sMCwithDodson', '20260326_Kanga_OFC_31turns_SRwithDodson',
                        '20260330_Kanga_OFC_30dot5turns_SRwithDodson',   '20260331_Kanga_OFC_30dot5turns_1sMCwithDodson',
                        '20260403_Kanga_OFC_30dot5turns_1sMCwithDodson', '20260406_Kanga_OFC_30dot5turns_1sMCwithDodson',
                        '20260406_Kanga_OFC_30dot5turns_SRwithDodson',
            
                        # dannon kanga
                        '20260409_Kanga_OFC_30dot5turns_1sMCwithDannon', '20260410_Kanga_OFC_30dot5turns_1sMCwithDannon',
                        '20260410_Kanga_OFC_30dot5turns_SRwithDannon',   '20260413_Kanga_OFC_30dot5turns_1sMCwithDannon',
                        '20260421_Kanga_OFC_30dot5turns_1sMCwithDannon',
                    ]
        task_conditions = [
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 'MC', 'SR', 'MC', 'SR',
                        'MC', 'SR', 'MC', 'SR', 'SR', 'MC',
                        'MC', 'MC', 'SR',
            
                        'MC', 'MC', 'SR', 'MC', 'MC',
                    ]
        dates_list = [
                        '20260309', '20260309_SR', '20260310', '20260310_SR', '20260311',    '20260311_SR',
                        '20260312', '20260312_SR', '20260313', '20260313_SR', '20260317',    '20260317_SR',
                        '20260318', '20260318_SR', '20260319', '20260319_SR', '20260323',    '20260323_SR',
                        '20260324', '20260324_SR', '20260326', '20260326_SR', '20260330_SR', '20260331',
                        '20260403', '20260406', '20260406_SR',
                    
                        '20260409', '20260410', '20260410_SR', '20260413',    '20260421',
                    ]
        videodates_list = [
                        '20260309', '20260309_SR', '20260310', '20260310_SR', '20260311',    '20260311_SR',
                        '20260312', '20260312_SR', '20260313', '20260313_SR', '20260317',    '20260317_SR',
                        '20260318', '20260318_SR', '20260319', '20260319_SR', '20260323',    '20260323_SR',
                        '20260324', '20260324_SR', '20260326', '20260326_SR', '20260330_SR', '20260331',
                        '20260403', '20260406', '20260406_SR', 
            
                        '20260409', '20260410', '20260410_SR', '20260413',    '20260421',
                    ] 
        
        session_start_times = [
                         0.00, 0.00, 0.00, 0.00, 0.00,  0.00, 
                         0.00, 0.00, 0.00, 0.00, 0.00,  0.00, 
                         0.00, 0.00, 0.00, 0.00, 0.00, 129.5,
                        116.2, 0.00, 0.00, 0.00, 0.00, 49.50,
                         0.00, 0.00, 0.00,
            
                         0.00, 0.00, 0.00, 0.00, 0.00,
                    ]
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = [ 'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0', 
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0', 
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0', 
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0',
                             
                              'Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0','Dev1/ai0',
                              ]
        animal1_fixedorders = [
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson',
                               'dodson','dodson','dodson','dodson','dodson','dodson', 
                               'dodson','dodson','dodson',
             
                               'dannon','dannon','dannon','dannon','dannon',
                              ]
        animal2_fixedorders = [
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 'kanga', 
                               'kanga', 'kanga', 'kanga', 
            
                               'kanga', 'kanga', 'kanga', 'kanga', 'kanga',
                              ]
        recordedanimals = animal2_fixedorders 

        animal1_filenames = [
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson","Dodson","Dodson","Dodson",
                             "Dodson","Dodson","Dodson",
            
                             "Dannon","Dannon","Dannon","Dannon","Dannon",
                            ]
        animal2_filenames = [
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga',
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 
                             'Kanga', 'Kanga', 'Kanga', 
            
                             'Kanga', 'Kanga', 'Kanga', 'Kanga', 'Kanga', 
                            ]
    

    
# a test case
if 1:
    if do_DLPFC:
        if 1: # kanga example
            neural_record_conditions = ['20240606_Kanga_MC']
            dates_list = ["20240606"]
            videodates_list = dates_list
            task_conditions = ['MC']
            session_start_times = [89] # in second
            kilosortvers = [4]
            trig_channelnames = ['Dev1/ai0']
            animal1_fixedorders = ['dannon']
            animal2_fixedorders = ['kanga']
            recordedanimals = animal2_fixedorders
            animal1_filenames = ["Dannon"]
            animal2_filenames = ["Kanga"]
        if 0: # dodson example 
            neural_record_conditions = ['20250415_Dodson_MC_withKanga']
            dates_list = ["20250415"]
            videodates_list = dates_list
            task_conditions = ['MC_withKanga']
            session_start_times = [363] # in second
            kilosortvers = [4]
            trig_channelnames = ['Dev1/ai0']
            animal1_fixedorders = ['dodson']
            recordedanimals = animal1_fixedorders
            animal2_fixedorders = ['kanga']
            animal1_filenames = ["Dodson"]
            animal2_filenames = ["Kanga"]
    #
    elif do_OFC:
        if 1: # kanga example
            neural_record_conditions = [ '20260309_Kanga_OFC_31turns_1sMCwithDodson',]
            task_conditions = [ 'MC', ]
            dates_list = [ '20260309', ]
            videodates_list = [ '20260309', ] 
            session_start_times = [0.00, ]
            kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
            trig_channelnames = [ 'Dev1/ai0',]
            animal1_fixedorders = ['dodson',]
            animal2_fixedorders = [ 'kanga',]
            recordedanimals = animal2_fixedorders 
            animal1_filenames = [ "Dodson",]
            animal2_filenames = ['Kanga', ]
        if 0: # dodson example
            neural_record_conditions = [ '20260309_Dodson_OFC_33turns_1sMCwithKanga',]
            task_conditions = [ 'MC', ]
            dates_list = [ '20260309', ]
            videodates_list = [ '20260309', ] 
            session_start_times = [0.00, ]
            kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
            trig_channelnames = [ 'Dev1/ai9',]
            animal1_fixedorders = ['dodson',]
            animal2_fixedorders = [ 'kanga',]
            recordedanimals = animal1_fixedorders 
            animal1_filenames = [ "Dodson",]
            animal2_filenames = ['Kanga', ]
    
    

ndates = np.shape(dates_list)[0]

session_start_frames = session_start_times * fps # fps is 30Hz

totalsess_time = 600

# video tracking results info
animalnames_videotrack = ['dodson','scorch'] # does not really mean dodson and scorch, instead, indicate animal1 and animal2
bodypartnames_videotrack = ['rightTuft','whiteBlaze','leftTuft','rightEye','leftEye','mouth']


# which camera to analyzed
cameraID = 'camera-2'
cameraID_short = 'cam2'

considerlevertube = 1
considertubeonly = 0

# location of levers and tubes for camera 2
# # camera 1
# lever_locs_camI = {'dodson':np.array([645,600]),'scorch':np.array([425,435])}
# tube_locs_camI  = {'dodson':np.array([1350,630]),'scorch':np.array([555,345])}
# # camera 2
# # location of the estimiated middle of the box
# lever_locs_camI = {'dodson':np.array([1325,615]),'scorch':np.array([560,615])}
# # location of the estimated lever
lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
tube_locs_camI  = {'dodson':np.array([1550,515]),'scorch':np.array([350,515])}
# # old
# # lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
# # tube_locs_camI  = {'dodson':np.array([1650,490]),'scorch':np.array([250,490])}
# # camera 3
# lever_locs_camI = {'dodson':np.array([1580,440]),'scorch':np.array([1296,540])}
# tube_locs_camI  = {'dodson':np.array([1470,375]),'scorch':np.array([805,475])}


if np.shape(session_start_times)[0] != np.shape(dates_list)[0]:
    exit()

    
# define bhv events summarizing variables     
tasktypes_all_dates = np.zeros((ndates,1))
coopthres_all_dates = np.zeros((ndates,1))

succ_rate_all_dates = np.zeros((ndates,1))
interpullintv_all_dates = np.zeros((ndates,1))
trialnum_all_dates = np.zeros((ndates,1))
totalsessiontime_all_dates = np.zeros((ndates,1))

owgaze1_num_all_dates = np.zeros((ndates,1))
owgaze2_num_all_dates = np.zeros((ndates,1))
mtgaze1_num_all_dates = np.zeros((ndates,1))
mtgaze2_num_all_dates = np.zeros((ndates,1))
pull1_num_all_dates = np.zeros((ndates,1))
pull2_num_all_dates = np.zeros((ndates,1))

bhv_intv_all_dates = dict.fromkeys(dates_list, [])

spike_trig_events_all_dates = dict.fromkeys(dates_list, [])

bhvevents_aligned_FR_all_dates = dict.fromkeys(dates_list, [])
bhvevents_aligned_FR_allevents_all_dates = dict.fromkeys(dates_list, [])

strategy_aligned_FR_all_dates = dict.fromkeys(dates_list, [])
strategy_aligned_FR_allevents_all_dates = dict.fromkeys(dates_list, [])

bhvevents_aligned_FRPCs_all_dates = dict.fromkeys(dates_list, [])
bhvevents_aligned_FRPCs_allevents_all_dates = dict.fromkeys(dates_list, [])

# where to save the summarizing data
data_saved_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/3d_recontruction_analysis_self_and_coop_task_data_saved/'

# neural data folder
if not do_OFC:
    neural_data_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/Marmoset_neural_recording/'
elif do_OFC:
    neural_data_folder = '/gpfs/marilyn/pi/nandy/Marmoset_neural_recording/'

    

In [ ]:
print(np.shape(neural_record_conditions))
print(np.shape(task_conditions))
print(np.shape(dates_list))
print(np.shape(videodates_list)) 
print(np.shape(session_start_times))

print(np.shape(kilosortvers))

print(np.shape(trig_channelnames))
print(np.shape(animal1_fixedorders)) 
print(np.shape(recordedanimals))
print(np.shape(animal2_fixedorders))

print(np.shape(animal1_filenames))
print(np.shape(animal2_filenames))  

In [ ]:
# basic behavior analysis (define time stamps for each bhv events, etc)

try:
    if redo_anystep:
        dummy
    
    # load saved data
    data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
    
    with open(data_saved_subfolder+'/owgaze1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        owgaze1_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/owgaze2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        owgaze2_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/mtgaze1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        mtgaze1_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/mtgaze2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        mtgaze2_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        pull1_num_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/pull2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        pull2_num_all_dates = pickle.load(f)

    with open(data_saved_subfolder+'/tasktypes_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        tasktypes_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/coopthres_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        coopthres_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/succ_rate_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        succ_rate_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/interpullintv_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        interpullintv_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/trialnum_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        trialnum_all_dates = pickle.load(f)
    with open(data_saved_subfolder+'/bhv_intv_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        bhv_intv_all_dates = pickle.load(f)

    with open(data_saved_subfolder+'/spike_trig_events_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        spike_trig_events_all_dates = pickle.load(f) 
        
    with open(data_saved_subfolder+'/bhvevents_aligned_FR_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        bhvevents_aligned_FR_all_dates = pickle.load(f) 
    with open(data_saved_subfolder+'/bhvevents_aligned_FR_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        bhvevents_aligned_FR_allevents_all_dates = pickle.load(f) 
        
    if do_OFC:
        with open(data_saved_subfolder+'/bhvevents_aligned_FRPCs_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
            bhvevents_aligned_FRPCs_all_dates = pickle.load(f) 
        with open(data_saved_subfolder+'/bhvevents_aligned_FRPCs_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
            bhvevents_aligned_FRPCs_allevents_all_dates = pickle.load(f) 
        
    if do_OFC:
        with open(data_saved_subfolder+'/totalsessiontime_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
            totalsessiontime_all_dates = pickle.load(f)
        
    print('all data from all dates are loaded')

except:

    print('analyze all dates')

    for idate in np.arange(0,ndates,1):
    
        date_tgt = dates_list[idate]
        videodate_tgt = videodates_list[idate]
        
        neural_record_condition = neural_record_conditions[idate]
        
        session_start_time = session_start_times[idate]
        
        kilosortver = kilosortvers[idate]
        
        trig_channelname = trig_channelnames[idate]
        
        animal1_filename = animal1_filenames[idate]
        animal2_filename = animal2_filenames[idate]
        
        animal1_fixedorder = [animal1_fixedorders[idate]]
        animal2_fixedorder = [animal2_fixedorders[idate]]
        
        recordedanimal = recordedanimals[idate]

        # folder and file path
        if not do_OFC:
            camera12_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_DLPFCs_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera12/"
            camera23_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_DLPFCs_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera23/"
        elif do_OFC:
            camera12_analyzed_path = "/gpfs/marilyn/pi/nandy/VideoTracker_SocialInter/test_video_cooperative_task_OFCs_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera12/"
            camera23_analyzed_path = "/gpfs/marilyn/pi/nandy/VideoTracker_SocialInter/test_video_cooperative_task_OFCs_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera23/"
        # 
        try: 
            singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
            bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"                
            # get the bodypart data from files
            bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
            video_file_original = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"
        except:
            singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
            bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            if not os.path.exists(bodyparts_camI_camIJ):
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            
            # get the bodypart data from files
            bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
            video_file_original = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"        
        
        # load behavioral results
        if not do_OFC:
            try:
                bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_cooperation_task_DLPFCs/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
                trial_record_json = glob.glob(bhv_data_path +date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_TrialRecord_" + "*.json")
                bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_bhv_data_" + "*.json")
                session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_session_info_" + "*.json")
                ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_ni_data_" + "*.json")
                #
                trial_record = pd.read_json(trial_record_json[0])
                bhv_data = pd.read_json(bhv_data_json[0])
                session_info = pd.read_json(session_info_json[0])
                # 
                with open(ni_data_json[0]) as f:
                    for line in f:
                        ni_data=json.loads(line)   
            except:
                bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_cooperation_task_DLPFCs/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
                trial_record_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_TrialRecord_" + "*.json")
                bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_bhv_data_" + "*.json")
                session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_session_info_" + "*.json")
                ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_ni_data_" + "*.json")
                #
                trial_record = pd.read_json(trial_record_json[0])
                bhv_data = pd.read_json(bhv_data_json[0])
                session_info = pd.read_json(session_info_json[0])
                #
                with open(ni_data_json[0]) as f:
                    for line in f:
                        ni_data=json.loads(line)
        
        elif do_OFC:
            try:
                bhv_data_path = "/gpfs/marilyn/pi/nandy/VideoTracker_SocialInter/marmoset_tracking_bhv_data_cooperation_task_OFCs/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
                trial_record_json = glob.glob(bhv_data_path +date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_TrialRecord_" + "*.json")
                bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_bhv_data_" + "*.json")
                session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_session_info_" + "*.json")
                ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_ni_data_" + "*.json")
                #
                trial_record = pd.read_json(trial_record_json[0])
                bhv_data = pd.read_json(bhv_data_json[0])
                session_info = pd.read_json(session_info_json[0])
                # 
                with open(ni_data_json[0]) as f:
                    for line in f:
                        ni_data=json.loads(line)   
            except:
                bhv_data_path = "/gpfs/marilyn/pi/nandy/VideoTracker_SocialInter/marmoset_tracking_bhv_data_cooperation_task_OFCs/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
                trial_record_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_TrialRecord_" + "*.json")
                bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_bhv_data_" + "*.json")
                session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_session_info_" + "*.json")
                ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_ni_data_" + "*.json")
                #
                trial_record = pd.read_json(trial_record_json[0])
                bhv_data = pd.read_json(bhv_data_json[0])
                session_info = pd.read_json(session_info_json[0])
                #
                with open(ni_data_json[0]) as f:
                    for line in f:
                        ni_data=json.loads(line)

            
        # get animal info from the session information
        animal1 = session_info['lever1_animal'][0].lower()
        animal2 = session_info['lever2_animal'][0].lower()

        
        # get task type and cooperation threshold
        try:
            coop_thres = session_info["pulltime_thres"][0]
            tasktype = session_info["task_type"][0]
        except:
            coop_thres = 0
            tasktype = 1
        tasktypes_all_dates[idate] = tasktype
        coopthres_all_dates[idate] = coop_thres   

        # clean up the trial_record
        warnings.filterwarnings('ignore')
        trial_record_clean = pd.DataFrame(columns=trial_record.columns)
        # for itrial in np.arange(0,np.max(trial_record['trial_number']),1):
        for itrial in trial_record['trial_number']:
            # trial_record_clean.loc[itrial] = trial_record[trial_record['trial_number']==itrial+1].iloc[[0]]
            trial_record_clean = trial_record_clean.append(trial_record[trial_record['trial_number']==itrial].iloc[[0]])
        trial_record_clean = trial_record_clean.reset_index(drop = True)

        # change bhv_data time to the absolute time
        time_points_new = pd.DataFrame(np.zeros(np.shape(bhv_data)[0]),columns=["time_points_new"])
        # for itrial in np.arange(0,np.max(trial_record_clean['trial_number']),1):
        for itrial in np.arange(0,np.shape(trial_record_clean)[0],1):
            # ind = bhv_data["trial_number"]==itrial+1
            ind = bhv_data["trial_number"]==trial_record_clean['trial_number'][itrial]
            new_time_itrial = bhv_data[ind]["time_points"] + trial_record_clean["trial_starttime"].iloc[itrial]
            time_points_new["time_points_new"][ind] = new_time_itrial
        bhv_data["time_points"] = time_points_new["time_points_new"]
        bhv_data = bhv_data[bhv_data["time_points"] != 0]


        # analyze behavior results
        # succ_rate_all_dates[idate] = np.sum(trial_record_clean["rewarded"]>0)/np.shape(trial_record_clean)[0]
        succ_rate_all_dates[idate] = np.sum((bhv_data['behavior_events']==3)|(bhv_data['behavior_events']==4))/np.sum((bhv_data['behavior_events']==1)|(bhv_data['behavior_events']==2))
        trialnum_all_dates[idate] = np.shape(trial_record_clean)[0]
        #
        pullid = np.array(bhv_data[(bhv_data['behavior_events']==1) | (bhv_data['behavior_events']==2)]["behavior_events"])
        pulltime = np.array(bhv_data[(bhv_data['behavior_events']==1) | (bhv_data['behavior_events']==2)]["time_points"])
        pullid_diff = np.abs(pullid[1:] - pullid[0:-1])
        pulltime_diff = pulltime[1:] - pulltime[0:-1]
        interpull_intv = pulltime_diff[pullid_diff==1]
        interpull_intv = interpull_intv[interpull_intv<10]
        mean_interpull_intv = np.nanmean(interpull_intv)
        std_interpull_intv = np.nanstd(interpull_intv)
        #
        interpullintv_all_dates[idate] = mean_interpull_intv
        # 
        if np.isin(animal1,animal1_fixedorder):
            pull1_num_all_dates[idate] = np.sum(bhv_data['behavior_events']==1) 
            pull2_num_all_dates[idate] = np.sum(bhv_data['behavior_events']==2)
        else:
            pull1_num_all_dates[idate] = np.sum(bhv_data['behavior_events']==2) 
            pull2_num_all_dates[idate] = np.sum(bhv_data['behavior_events']==1)

        
        # load behavioral event results
        try:
            # dummy
            print('load social gaze with '+cameraID+' only of '+date_tgt)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'rb') as f:
                output_look_ornot = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'rb') as f:
                output_allvectors = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'rb') as f:
                output_allangles = pickle.load(f)  
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'rb') as f:
                output_key_locations = pickle.load(f)
                
        except:   
            print('analyze social gaze with '+cameraID+' only of '+date_tgt)
            # get social gaze information 
            output_look_ornot, output_allvectors, output_allangles = find_socialgaze_timepoint_singlecam_wholebody(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,
                                                                                                                   considerlevertube,considertubeonly,sqr_thres_tubelever,
                                                                                                                   sqr_thres_face,sqr_thres_body)
            
            output_key_locations = find_socialgaze_timepoint_singlecam_wholebody_2(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,considerlevertube)
            
            # save data
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            #
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'wb') as f:
                pickle.dump(output_look_ornot, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'wb') as f:
                pickle.dump(output_allvectors, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'wb') as f:
                pickle.dump(output_allangles, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'wb') as f:
                pickle.dump(output_key_locations, f)
  

        look_at_other_or_not_merge = output_look_ornot['look_at_other_or_not_merge']
        look_at_tube_or_not_merge = output_look_ornot['look_at_tube_or_not_merge']
        look_at_lever_or_not_merge = output_look_ornot['look_at_lever_or_not_merge']
        # change the unit to second and align to the start of the session
        session_start_time = session_start_times[idate]
        look_at_other_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_other_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_lever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_lever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_tube_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_tube_or_not_merge['dodson'])[0],1)/fps - session_start_time 

        # find time point of behavioral events
        output_time_points_socialgaze ,output_time_points_levertube = bhv_events_timepoint_singlecam(bhv_data,look_at_other_or_not_merge,look_at_lever_or_not_merge,look_at_tube_or_not_merge)
        time_point_pull1 = output_time_points_socialgaze['time_point_pull1']
        time_point_pull2 = output_time_points_socialgaze['time_point_pull2']
        oneway_gaze1 = output_time_points_socialgaze['oneway_gaze1']
        oneway_gaze2 = output_time_points_socialgaze['oneway_gaze2']
        mutual_gaze1 = output_time_points_socialgaze['mutual_gaze1']
        mutual_gaze2 = output_time_points_socialgaze['mutual_gaze2']
        # 
        # mostly just for the sessions in which MC and SR are in the same session 
        firstpulltime = np.nanmin([np.nanmin(time_point_pull1),np.nanmin(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1>(firstpulltime-15)] # 15s before the first pull (animal1 or 2) count as the active period
        oneway_gaze2 = oneway_gaze2[oneway_gaze2>(firstpulltime-15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1>(firstpulltime-15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2>(firstpulltime-15)]  
        #    
        # newly added condition: only consider gaze during the active pulling time (15s after the last pull)    
        lastpulltime = np.nanmax([np.nanmax(time_point_pull1),np.nanmax(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1<(lastpulltime+15)]    
        oneway_gaze2 = oneway_gaze2[oneway_gaze2<(lastpulltime+15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1<(lastpulltime+15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2<(lastpulltime+15)] 
            
        # define successful pulls and failed pulls
        if 0: # old definition; not in use
            trialnum_succ = np.array(trial_record_clean['trial_number'][trial_record_clean['rewarded']>0])
            bhv_data_succ = bhv_data[np.isin(bhv_data['trial_number'],trialnum_succ)]
            #
            time_point_pull1_succ = bhv_data_succ["time_points"][bhv_data_succ["behavior_events"]==1]
            time_point_pull2_succ = bhv_data_succ["time_points"][bhv_data_succ["behavior_events"]==2]
            time_point_pull1_succ = np.round(time_point_pull1_succ,1)
            time_point_pull2_succ = np.round(time_point_pull2_succ,1)
            #
            trialnum_fail = np.array(trial_record_clean['trial_number'][trial_record_clean['rewarded']==0])
            bhv_data_fail = bhv_data[np.isin(bhv_data['trial_number'],trialnum_fail)]
            #
            time_point_pull1_fail = bhv_data_fail["time_points"][bhv_data_fail["behavior_events"]==1]
            time_point_pull2_fail = bhv_data_fail["time_points"][bhv_data_fail["behavior_events"]==2]
            time_point_pull1_fail = np.round(time_point_pull1_fail,1)
            time_point_pull2_fail = np.round(time_point_pull2_fail,1)
        else:
            # a new definition of successful and failed pulls
            # separate successful and failed pulls
            # step 1 all pull and juice
            time_point_pull1 = bhv_data["time_points"][bhv_data["behavior_events"]==1]
            time_point_pull2 = bhv_data["time_points"][bhv_data["behavior_events"]==2]
            time_point_juice1 = bhv_data["time_points"][bhv_data["behavior_events"]==3]
            time_point_juice2 = bhv_data["time_points"][bhv_data["behavior_events"]==4]
            # step 2:
            # pull 1
            # Find the last pull before each juice
            successful_pull1 = [time_point_pull1[time_point_pull1 < juice].max() for juice in time_point_juice1]
            # Convert to Pandas Series
            successful_pull1 = pd.Series(successful_pull1, index=time_point_juice1.index)
            # Find failed pulls (pulls that are not successful)
            failed_pull1 = time_point_pull1[~time_point_pull1.isin(successful_pull1)]
            # pull 2
            # Find the last pull before each juice
            successful_pull2 = [time_point_pull2[time_point_pull2 < juice].max() for juice in time_point_juice2]
            # Convert to Pandas Series
            successful_pull2 = pd.Series(successful_pull2, index=time_point_juice2.index)
            # Find failed pulls (pulls that are not successful)
            failed_pull2 = time_point_pull2[~time_point_pull2.isin(successful_pull2)]
            #
            # step 3:
            time_point_pull1_succ = np.round(successful_pull1,1)
            time_point_pull2_succ = np.round(successful_pull2,1)
            time_point_pull1_fail = np.round(failed_pull1,1)
            time_point_pull2_fail = np.round(failed_pull2,1)
        # 
        time_point_pulls_succfail = { "pull1_succ":time_point_pull1_succ,
                                      "pull2_succ":time_point_pull2_succ,
                                      "pull1_fail":time_point_pull1_fail,
                                      "pull2_fail":time_point_pull2_fail,
                                    }
            
        # new total session time (instead of 600s) - total time of the video recording
        totalsess_time = np.floor(np.shape(output_look_ornot['look_at_lever_or_not_merge']['dodson'])[0]/30) 
                
        totalsessiontime_all_dates[idate] = totalsess_time - session_start_time    
        
        # # plot behavioral events
        if 0:
            if np.isin(animal1,animal1_fixedorder):
                    plot_bhv_events(date_tgt,animal1, animal2, session_start_time, totalsess_time, time_point_pull1, time_point_pull2, oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2)
            else:
                    plot_bhv_events(date_tgt,animal2, animal1, session_start_time, totalsess_time, time_point_pull2, time_point_pull1, oneway_gaze2, oneway_gaze1, mutual_gaze2, mutual_gaze1)
            #
            # save behavioral events plot
            if 0:
                current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]
                add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
                if not os.path.exists(add_date_dir):
                    os.makedirs(add_date_dir)
                plt.savefig(data_saved_folder+"/bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/'+date_tgt+"_"+cameraID_short+".pdf")

        #
        if np.isin(animal1,animal1_fixedorder):
            owgaze1_num_all_dates[idate] = np.shape(oneway_gaze1)[0]
            owgaze2_num_all_dates[idate] = np.shape(oneway_gaze2)[0]
            mtgaze1_num_all_dates[idate] = np.shape(mutual_gaze1)[0]
            mtgaze2_num_all_dates[idate] = np.shape(mutual_gaze2)[0]
        else:            
            owgaze1_num_all_dates[idate] = np.shape(oneway_gaze2)[0]
            owgaze2_num_all_dates[idate] = np.shape(oneway_gaze1)[0]
            mtgaze1_num_all_dates[idate] = np.shape(mutual_gaze2)[0]
            mtgaze2_num_all_dates[idate] = np.shape(mutual_gaze1)[0]

     
        # get the continuous variables
        gausKernelsize = 16 # 4 or 16
        #
        data_summary_twoanimals, data_summary_names = get_data_for_singlecam_conBhv_from_highDimension_to_PCspace(gausKernelsize, fps, animal1, animal2, 
                                                    animalnames_videotrack, session_start_time, 
                                                    time_point_pull1, time_point_pull2,
                                                    time_point_juice1, time_point_juice2, 
                                                    oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2, 
                                                    output_look_ornot, output_allvectors, 
                                                    output_allangles, output_key_locations)
        # make sure the two animals' tracking result have the same size
        # Extract the data for cleaner reference
        data1 = data_summary_twoanimals[animal1]
        data2 = data_summary_twoanimals[animal2]
        # 1. Find the lengths of the 'xxx' dimension by checking the first row
        len1 = len(data1[0])
        len2 = len(data2[0])
        max_len = max(len1, len2)
        # 2. Apply padding based on the data type

        # --- METHOD A: If they are 2D NumPy Arrays ---
        if isinstance(data1, np.ndarray):
            # np.pad allows us to specify padding for ((rows_before, rows_after), (cols_before, cols_after))
            # We add 0 padding to the 17 rows, and pad the difference to the end of the columns.
            if len1 < max_len:
                data_summary_twoanimals[animal1] = np.pad(data1, ((0, 0), (0, max_len - len1)), constant_values=np.nan)
            elif len2 < max_len:
                data_summary_twoanimals[animal2] = np.pad(data2, ((0, 0), (0, max_len - len2)), constant_values=np.nan)
        # --- METHOD B: If they are standard Python lists of 1D arrays ---
        else:
            def pad_list_of_arrays(data_list, target_len):
                padded_list = []
                for arr in data_list:
                    # Create an array of NaNs for the missing length
                    nans = np.full(target_len - len(arr), np.nan)
                    # Concatenate the original array with the NaNs
                    padded_list.append(np.concatenate((arr, nans)))
                return padded_list

            if len1 < max_len:
                data_summary_twoanimals[animal1] = pad_list_of_arrays(data1, max_len)
            elif len2 < max_len:
                data_summary_twoanimals[animal2] = pad_list_of_arrays(data2, max_len)

        # the pca on the continuous bhv for each animal
        #
        vars_toPCA_names = ['gaze_other_angle', 'gaze_tube_angle', 'gaze_lever_angle', 'animal_animal_dist',
                            'animal_tube_dist', 'animal_lever_dist', 'mass_move_speed', 'gaze_angle_speed',]
        #
        indices = [data_summary_names.index(name) for name in vars_toPCA_names]
        # 
        # ==========================================
        # PCA FOR ANIMAL 1
        # ==========================================
        allbhvs_a1 = np.array(data_summary_twoanimals[animal1])[indices,:]
        data_for_pca_a1 = allbhvs_a1.T
        # 1. Create a mask to find valid rows (no NaNs)
        valid_mask_a1 = ~np.isnan(data_for_pca_a1).any(axis=1)
        valid_data_a1 = data_for_pca_a1[valid_mask_a1]
        # 2. Normalize and run PCA ONLY on valid data
        scaler_a1 = StandardScaler()
        data_scaled_a1 = scaler_a1.fit_transform(valid_data_a1)
        pca_a1 = PCA(n_components=3)
        principal_components_valid_a1 = pca_a1.fit_transform(data_scaled_a1)
        explained_variance_a1 = pca_a1.explained_variance_ratio_
        # 3. Reconstruct the full array (restoring the NaNs at the end)
        principal_components_a1 = np.full((data_for_pca_a1.shape[0], 3), np.nan)
        principal_components_a1[valid_mask_a1] = principal_components_valid_a1
        principal_components_transposed_a1 = principal_components_a1.T
        PC1_a1 = principal_components_transposed_a1[0,:]
        #
        # ==========================================
        # PCA FOR ANIMAL 2
        # ==========================================
        allbhvs_a2 = np.array(data_summary_twoanimals[animal2])[indices,:]
        data_for_pca_a2 = allbhvs_a2.T
        # 1. Create a mask to find valid rows (no NaNs)
        valid_mask_a2 = ~np.isnan(data_for_pca_a2).any(axis=1)
        valid_data_a2 = data_for_pca_a2[valid_mask_a2]
        # 2. Normalize and run PCA ONLY on valid data
        scaler_a2 = StandardScaler()
        data_scaled_a2 = scaler_a2.fit_transform(valid_data_a2)
        pca_a2 = PCA(n_components=3)
        principal_components_valid_a2 = pca_a2.fit_transform(data_scaled_a2)
        explained_variance_a2 = pca_a2.explained_variance_ratio_
        # 3. Reconstruct the full array (restoring the NaNs at the end)
        principal_components_a2 = np.full((data_for_pca_a2.shape[0], 3), np.nan)
        principal_components_a2[valid_mask_a2] = principal_components_valid_a2
        principal_components_transposed_a2 = principal_components_a2.T
        PC1_a2 = principal_components_transposed_a2[0,:]
        #
        # ==========================================
        # APPEND RESULTS
        # ==========================================
        # Note: If data_summary_twoanimals is a pure 2D np.ndarray at this point,
        # .append() will fail. If it is a list of arrays, this works perfectly.
        data_summary_twoanimals[animal1].append(PC1_a1)
        data_summary_twoanimals[animal2].append(PC1_a2)
        data_summary_twoanimals[animal1].append(PC1_a2)
        data_summary_twoanimals[animal2].append(PC1_a1)
        #
        data_summary_names.append('self_PC1')
        data_summary_names.append('other_PC1')
        #
        # add gaze filtered other_pc1 as the social evidence
        # animal 1
        ind_socialgaze = [data_summary_names.index(var) for var in ['socialgaze_prob']][0]
        socialgaze_filter = (data_summary_twoanimals[animal1][ind_socialgaze]>\
                             np.nanmin(data_summary_twoanimals[animal1][ind_socialgaze])).astype(int)
        ind_otherPC1 = [data_summary_names.index(var) for var in ['other_PC1']][0]
        socialevidence_a1 = data_summary_twoanimals[animal1][ind_otherPC1]*socialgaze_filter
        # animal 2
        ind_socialgaze = [data_summary_names.index(var) for var in ['socialgaze_prob']][0]
        socialgaze_filter = (data_summary_twoanimals[animal2][ind_socialgaze]>\
                             np.nanmin(data_summary_twoanimals[animal2][ind_socialgaze])).astype(int)
        ind_otherPC1 = [data_summary_names.index(var) for var in ['other_PC1']][0]
        socialevidence_a2 = data_summary_twoanimals[animal2][ind_otherPC1]*socialgaze_filter
        # APPEND RESULTS
        data_summary_twoanimals[animal1].append(socialevidence_a1)
        data_summary_twoanimals[animal2].append(socialevidence_a2)
        #
        data_summary_names.append('social_evidence')
        
        
       
        #    

        
        
        # # load spike sorting results
        if 1:
            
            
            # session starting time compared with the neural recording
            session_start_time_niboard_offset = ni_data['session_t0_offset'] # in the unit of second
            try:
                neural_start_time_niboard_offset = ni_data['trigger_ts'][0]['elapsed_time'] # in the unit of second
            except: # for the multi-animal recording setup
                neural_start_time_niboard_offset = next(
                    entry['timepoints'][0]['elapsed_time']
                    for entry in ni_data['trigger_ts']
                    if entry['channel_name'] == f"{trig_channelname}")
            neural_start_time_session_start_offset = neural_start_time_niboard_offset-session_start_time_niboard_offset

            # load channel maps
            channel_map_file = '/home/ws523/kilisort_spikesorting/Channel-Maps/Neuronexus_whitematter_2x32.mat'
            # channel_map_file = '/home/ws523/kilisort_spikesorting/Channel-Maps/Neuronexus_whitematter_2x32_kilosort4_new.mat'
            channel_map_data = scipy.io.loadmat(channel_map_file)
            
            
            # # load spike sorting results
            print('load spike data for '+neural_record_condition)
            if kilosortver == 2:
                spike_time_file = neural_data_folder+neural_record_condition+'/Kilosort/spike_times.npy'
                spike_time_data = np.load(spike_time_file)
            elif kilosortver == 4:
                spike_time_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/spike_times.npy'
                spike_time_data = np.load(spike_time_file)
            # 
            # align the FR recording time stamps
            spike_time_data = spike_time_data + fs_spikes*neural_start_time_session_start_offset
            # down-sample the spike recording resolution to 30Hz
            spike_time_data = spike_time_data/fs_spikes*fps
            spike_time_data = np.round(spike_time_data)
            #
            if kilosortver == 2:
                spike_clusters_file = neural_data_folder+neural_record_condition+'/Kilosort/spike_clusters.npy'
                spike_clusters_data = np.load(spike_clusters_file)
                spike_channels_data = np.copy(spike_clusters_data)
            elif kilosortver == 4:
                spike_clusters_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/spike_clusters.npy'
                spike_clusters_data = np.load(spike_clusters_file)
                spike_channels_data = np.copy(spike_clusters_data)
            #
            if kilosortver == 2:
                channel_maps_file = neural_data_folder+neural_record_condition+'/Kilosort/channel_map.npy'
                channel_maps_data = np.load(channel_maps_file)
            elif kilosortver == 4:
                channel_maps_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/channel_map.npy'
                channel_maps_data = np.load(channel_maps_file)
            #
            if kilosortver == 2:
                channel_pos_file = neural_data_folder+neural_record_condition+'/Kilosort/channel_positions.npy'
                channel_pos_data = np.load(channel_pos_file)
            elif kilosortver == 4:
                channel_pos_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/channel_positions.npy'
                channel_pos_data = np.load(channel_pos_file)
            #
            if kilosortver == 2:
                clusters_info_file = neural_data_folder+neural_record_condition+'/Kilosort/cluster_info.tsv'
                clusters_info_data = pd.read_csv(clusters_info_file,sep="\t")
            elif kilosortver == 4:
                clusters_info_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/cluster_info.tsv'
                clusters_info_data = pd.read_csv(clusters_info_file,sep="\t")
            #
            # only get the spikes that are manually checked
            try:
                good_clusters = clusters_info_data[(clusters_info_data.group=='good')|(clusters_info_data.group=='mua')]['cluster_id'].values
            except:
                good_clusters = clusters_info_data[(clusters_info_data.group=='good')|(clusters_info_data.group=='mua')]['id'].values
            #
            clusters_info_data = clusters_info_data[~pd.isnull(clusters_info_data.group)]
            #
            spike_time_data = spike_time_data[np.isin(spike_clusters_data,good_clusters)]
            spike_channels_data = spike_channels_data[np.isin(spike_clusters_data,good_clusters)]
            spike_clusters_data = spike_clusters_data[np.isin(spike_clusters_data,good_clusters)]
            
            #
            nclusters = np.shape(clusters_info_data)[0]
            #
            for icluster in np.arange(0,nclusters,1):
                try:
                    cluster_id = clusters_info_data['id'].iloc[icluster]
                except:
                    cluster_id = clusters_info_data['cluster_id'].iloc[icluster]
                spike_channels_data[np.isin(spike_clusters_data,cluster_id)] = clusters_info_data['ch'].iloc[icluster]   
            # 
            # get the channel to depth information, change 2 shanks to 1 shank 
            try:
                channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1]*2,channel_pos_data[channel_pos_data[:,0]==1,1]*2+1])
                # channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1],channel_pos_data[channel_pos_data[:,0]==31.2,1]])            
                # channel_to_depth = np.vstack([channel_maps_data.T[0],channel_depth])
                channel_to_depth = np.vstack([channel_maps_data.T,channel_depth])
            except:
                channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1],channel_pos_data[channel_pos_data[:,0]==31.2,1]])            
                # channel_to_depth = np.vstack([channel_maps_data.T[0],channel_depth])
                channel_to_depth = np.vstack([channel_maps_data.T,channel_depth])
                channel_to_depth[1] = channel_to_depth[1]/30-64 # make the y axis consistent
            #
           
            
            # calculate the firing rate
            # FR_kernel = 0.20 # in the unit of second
            FR_kernel = 1/30 # in the unit of second # 1/30 same resolution as the video recording
            # FR_kernel is sent to to be this if want to explore it's relationship with continuous trackng data
            
            # totalsess_time_forFR = np.floor(np.shape(output_look_ornot['look_at_lever_or_not_merge']['dodson'])[0]/30)  # to match the total time of the video recording
            totalsess_time_forFR = np.ceil(np.nanmax([np.nanmax(time_point_pull1), \
                                                      np.nanmax(time_point_pull2)])+session_start_time)+5 # only the functioning time (pulling time)
            
            _,FR_timepoint_allch,FR_allch,FR_zscore_allch = spike_analysis_FR_calculation(fps, FR_kernel, totalsess_time_forFR,
                                                                                          spike_clusters_data, spike_time_data)
            # _,FR_timepoint_allch,FR_allch,FR_zscore_allch = spike_analysis_FR_calculation(fps,FR_kernel,totalsess_time_forFR,
            #                                                                              spike_channels_data, spike_time_data)
            # behavioral events aligned firing rate for each unit
            if 0: 
                print('plot event aligned firing rate')
                #
                savefig = 1
                save_path = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents/"+cameraID+"/"+animal1_filename+"_"+animal2_filename+"_"+recordedanimal+"Recorded/"+date_tgt
                if not os.path.exists(save_path):
                    os.makedirs(save_path)
                #
                aligntwins = 4 # 5 second
                gaze_thresold = 0.2 # min length threshold to define if a gaze is real gaze or noise, in the unit of second 
                #
                bhvevents_aligned_FR_average_all,bhvevents_aligned_FR_allevents_all = plot_bhv_events_aligned_FR(date_tgt,savefig,save_path, animal1, animal2,time_point_pull1,time_point_pull2,time_point_pulls_succfail,
                                           oneway_gaze1,oneway_gaze2,mutual_gaze1,mutual_gaze2,gaze_thresold,totalsess_time_forFR,
                                           aligntwins,fps,FR_timepoint_allch,FR_zscore_allch,clusters_info_data)
                
                bhvevents_aligned_FR_all_dates[date_tgt] = bhvevents_aligned_FR_average_all
                bhvevents_aligned_FR_allevents_all_dates[date_tgt] = bhvevents_aligned_FR_allevents_all
                
            
            #
            # Run PCA analysis
            FR_zscore_allch_np_merged = np.array(pd.DataFrame(FR_zscore_allch).T)
            FR_zscore_allch_np_merged = FR_zscore_allch_np_merged[~np.isnan(np.sum(FR_zscore_allch_np_merged,axis=1)),:]
            # # run PCA on the entire session
            pca = PCA(n_components=3)
            FR_zscore_allch_PCs = pca.fit_transform(FR_zscore_allch_np_merged.T)
            #
            # plot and save the bhv event aligned PC 1,2,3 traces
            if 1:
                print('plot bhv aligned firing rate PC traces')
                #
                FRPCs_zscore_allch={'pc1':FR_zscore_allch_PCs[:,0],
                                    'pc2':FR_zscore_allch_PCs[:,1],
                                    'pc3':FR_zscore_allch_PCs[:,2],}
                #
                clusters_info_data_PCs = pd.DataFrame({'ch':['pc1','pc2','pc3'],'id':['pc1','pc2','pc3']})

                #
                savefig = 1
                save_path = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents/"+cameraID+"/"+\
                            animal1_filename+"_"+animal2_filename+"_"+recordedanimal+"Recorded/"+date_tgt+"/FR_PCtraces/"
                if not os.path.exists(save_path):
                    os.makedirs(save_path)
                #
                aligntwins = 4 # 5 second
                gaze_thresold = 0.2 # min length threshold to define if a gaze is real gaze or noise, in the unit of second 
                #
                bhvevents_aligned_FRPCs_average_all,bhvevents_aligned_FRPCs_allevents_all = plot_bhv_events_aligned_FR(date_tgt,savefig,save_path, animal1, animal2,time_point_pull1,time_point_pull2,time_point_pulls_succfail,
                                           oneway_gaze1,oneway_gaze2,mutual_gaze1,mutual_gaze2,gaze_thresold,totalsess_time_forFR,
                                           aligntwins,fps,FR_timepoint_allch,FRPCs_zscore_allch,clusters_info_data_PCs)
                
                bhvevents_aligned_FRPCs_all_dates[date_tgt] = bhvevents_aligned_FRPCs_average_all
                bhvevents_aligned_FRPCs_allevents_all_dates[date_tgt] = bhvevents_aligned_FRPCs_allevents_all
            
            
            
           
            

    # save data
    if 0:
        
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)
                
        # with open(data_saved_subfolder+'/DBN_input_data_alltypes_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
        #     pickle.dump(DBN_input_data_alltypes, f)

        with open(data_saved_subfolder+'/owgaze1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(owgaze1_num_all_dates, f)
        with open(data_saved_subfolder+'/owgaze2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(owgaze2_num_all_dates, f)
        with open(data_saved_subfolder+'/mtgaze1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(mtgaze1_num_all_dates, f)
        with open(data_saved_subfolder+'/mtgaze2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(mtgaze2_num_all_dates, f)
        with open(data_saved_subfolder+'/pull1_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(pull1_num_all_dates, f)
        with open(data_saved_subfolder+'/pull2_num_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(pull2_num_all_dates, f)

        with open(data_saved_subfolder+'/tasktypes_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(tasktypes_all_dates, f)
        with open(data_saved_subfolder+'/coopthres_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(coopthres_all_dates, f)
        with open(data_saved_subfolder+'/succ_rate_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(succ_rate_all_dates, f)
        with open(data_saved_subfolder+'/interpullintv_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(interpullintv_all_dates, f)
        with open(data_saved_subfolder+'/trialnum_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(trialnum_all_dates, f)
        with open(data_saved_subfolder+'/bhv_intv_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(bhv_intv_all_dates, f)
            
        with open(data_saved_subfolder+'/totalsessiontime_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(totalsessiontime_all_dates, f)
            
        with open(data_saved_subfolder+'/spike_trig_events_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(spike_trig_events_all_dates, f)  
    
        with open(data_saved_subfolder+'/bhvevents_aligned_FR_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(bhvevents_aligned_FR_all_dates, f) 
        with open(data_saved_subfolder+'/bhvevents_aligned_FR_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(bhvevents_aligned_FR_allevents_all_dates, f) 
            
    
    
    # only save a subset 
    if 0:
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)
    
        with open(data_saved_subfolder+'/bhvevents_aligned_FRPCs_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(bhvevents_aligned_FRPCs_all_dates, f) 
        with open(data_saved_subfolder+'/bhvevents_aligned_FRPCs_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(bhvevents_aligned_FRPCs_allevents_all_dates, f) 
    
    

In [ ]:
# data_summary_twoanimals

In [ ]:
# totalsess_time_forFR

In [ ]:
if 0:
    # testing code for running the inverse RL model to estimate the values
    # prepare the data

    import numpy as np
    import pandas as pd

    # =========================================================
    # 1. Configuration & Variable Mapping
    # =========================================================
    fps = 30
    downsample_rate = 3              # Downsample 30 Hz to 10 Hz
    window_seconds = 4
    frames_per_window = window_seconds * fps
    rolling_window_frames = 2 * fps  # 2.0-second trailing window for partner noise

    # Extract data specifically for the recorded animal
    data = data_summary_twoanimals[recordedanimal]
    var_names = data_summary_names

    # Map indices
    dist_idx = var_names.index('animal_lever_dist')
    gaze_idx = var_names.index('socialgaze_prob')
    pc1_idx = var_names.index('other_PC1')
    speed_idx = var_names.index('mass_move_speed')

    # Extract raw 1D session arrays
    lever_dist_session = data[dist_idx]
    gaze_prob_session = data[gaze_idx]
    speed_session = data[speed_idx]
    raw_other_pc1 = data[pc1_idx]

    # =========================================================
    # 2. Continuous Session Processing (Avoids Edge Effects)
    # =========================================================
    # Smooth over any camera tracking NaNs before calculating rolling variance
    clean_other_pc1 = pd.Series(raw_other_pc1).interpolate(method='linear').ffill().bfill()
    rolling_std_session = clean_other_pc1.rolling(window=rolling_window_frames, min_periods=1).std().values

    # =========================================================
    # 3. Extract Active Deliberation Windows (-4s to 0s)
    # =========================================================
    raw_windows = []
    all_trial_distances = []
    all_trial_noises = []
    all_trial_speeds = []

    for pull_t in selfpull_time:
        pull_idx = int(np.round((pull_t - session_start_time) * fps))
        start_idx = pull_idx - frames_per_window

        # Skip if window falls outside recorded boundaries
        if start_idx < 0 or pull_idx >= len(lever_dist_session):
            continue

        # Slice the continuous active window
        trial_dist = lever_dist_session[start_idx:pull_idx]
        trial_gaze = gaze_prob_session[start_idx:pull_idx]
        trial_noise = rolling_std_session[start_idx:pull_idx]
        trial_speed = speed_session[start_idx:pull_idx]

        raw_windows.append({
            'dist': trial_dist,
            'gaze': trial_gaze,
            'noise': trial_noise,
            'speed': trial_speed
        })

        # Pool data to calculate context-aware thresholds
        all_trial_distances.extend(trial_dist)
        all_trial_noises.extend(trial_noise)
        all_trial_speeds.extend(trial_speed)

    # =========================================================
    # 4. Calculate Dynamic Thresholds
    # =========================================================
    # Distance: Divide the actual approach paths into Close, Mid, and Far thirds
    dist_bins = np.percentile(all_trial_distances, [33.3, 66.6])
    # Noise & Speed: Use the median of the active cooperative periods
    noise_threshold = np.median(all_trial_noises)
    speed_threshold = np.median(all_trial_speeds)

    print(f"Dynamic Distance Bins: {np.round(dist_bins, 3)}")
    print(f"Noise Threshold: {round(noise_threshold, 3)} | Speed Threshold: {round(speed_threshold, 3)}")

    # =========================================================
    # 5. Discretize into MDP Trajectories
    # =========================================================
    trajectories = []

    for window in raw_windows:
        trial_states = []
        trial_actions = []

        # Step through the 4-second window at 10 Hz
        for i in range(0, frames_per_window, downsample_rate):

            # --- DEFINE STATE (S) ---
            s_d = np.digitize(window['dist'][i], bins=dist_bins) # 0=Close, 1=Mid, 2=Far
            s_g = 1 if window['gaze'][i] > 0.0 else 0            # 1=Looking, 0=Not Looking
            s_n = 1 if window['noise'][i] > noise_threshold else 0 # 1=Noisy, 0=Stable

            # Flatten into a unique State Index (0 to 11)
            state_idx = s_d * 4 + s_g * 2 + s_n
            trial_states.append(state_idx)

            # --- DEFINE ACTION (A) ---
            if i + downsample_rate >= frames_per_window:
                action = 3  # Terminal Action: Execute Pull
            else:
                # Predict action based on the kinematics of the *next* time step
                next_gaze = window['gaze'][i + downsample_rate]
                next_speed = window['speed'][i + downsample_rate]

                # Action Hierarchy captures the "Stop-and-Look" behavior
                if next_gaze > 0.0:
                    action = 2  # Sample (Cognitive effort prioritizes over movement)
                elif next_speed > speed_threshold:
                    action = 1  # Moving (Physical effort)
                else:
                    action = 0  # Wait (Stationary baseline)

            trial_actions.append(action)

            # NEW LINE: Add the Terminal Goal State that Action 3 transitions into
            trial_states.append(12)

        trajectories.append({'states': trial_states, 'actions': trial_actions})

    print(f"Successfully formatted {len(trajectories)} trial trajectories for IRL.")

In [ ]:
if 0:
    
    import numpy as np

    class MaxEntIRL:
        # Notice n_states is now 13 to account for our Terminal Goal State
        def __init__(self, trajectories, n_states=13, n_actions=4, gamma=0.95):
            self.trajectories = trajectories
            self.n_states = n_states
            self.n_actions = n_actions
            self.gamma = gamma
            self.P = self._estimate_transitions()

            # We only learn weights for the 12 behavioral states. 
            # We initialize them as small negative numbers (representing the cost of effort/time).
            self.theta = np.random.uniform(-1, 0, size=(12,))

        def _estimate_transitions(self):
            """Calculate P(s'|s,a) with the Terminal State included."""
            P = np.zeros((self.n_states, self.n_actions, self.n_states))
            counts = np.zeros((self.n_states, self.n_actions))

            for traj in self.trajectories:
                # Now loops through all 40 actions. The final action (3) will perfectly 
                # map the 40th state to the 41st state (State 12).
                for t in range(len(traj['actions'])):
                    s = traj['states'][t]
                    a = traj['actions'][t]
                    s_next = traj['states'][t+1]
                    P[s, a, s_next] += 1
                    counts[s, a] += 1

            # Normalize probabilities
            for s in range(self.n_states):
                for a in range(self.n_actions):
                    if counts[s, a] == 0:
                        P[s, a, :] = 1.0 / self.n_states
                    else:
                        P[s, a, :] /= counts[s, a]

            # State 12 is the Absorbing Goal State (once you pull, the trial is over)
            for a in range(self.n_actions):
                P[12, a, :] = 0.0
                P[12, a, 12] = 1.0 # You stay in State 12 forever

            return P

        def expert_feature_expectations(self):
            """Calculate state visitations, ignoring the terminal state for gradient updates."""
            feat_exp = np.zeros(12)
            for traj in self.trajectories:
                for t, s in enumerate(traj['states']):
                    if s < 12: # Only track the 12 behavioral states
                        feat_exp[s] += (self.gamma ** t)
            return feat_exp / len(self.trajectories)

        def value_iteration(self, reward):
            V = np.zeros(self.n_states)
            policy = np.zeros((self.n_states, self.n_actions))
            threshold = 1e-4

            while True:
                V_prev = np.copy(V)
                Q = np.zeros((self.n_states, self.n_actions))
                for s in range(self.n_states):
                    for a in range(self.n_actions):
                        Q[s, a] = reward[s] + self.gamma * np.sum(self.P[s, a, :] * V)

                max_Q = np.max(Q, axis=1, keepdims=True)
                exp_Q = np.exp(Q - max_Q)
                policy = exp_Q / np.sum(exp_Q, axis=1, keepdims=True)

                V = np.sum(policy * Q, axis=1)
                if np.max(np.abs(V - V_prev)) < threshold:
                    break
            return V, policy

        def state_visitation_frequencies(self, policy):
            mu = np.zeros(self.n_states)
            mu_initial = np.ones(self.n_states) / self.n_states 

            horizon = 40 
            for t in range(horizon):
                mu_t = np.zeros(self.n_states)
                for s in range(self.n_states):
                    for a in range(self.n_actions):
                        mu_t += mu[s] * policy[s, a] * self.P[s, a, :]
                mu += (self.gamma ** t) * mu_t
            return mu[:12] # Only return frequencies for the 12 behavioral states

        def train(self, epochs=200, learning_rate=0.05):
            expert_feat_exp = self.expert_feature_expectations()

            for epoch in range(epochs):
                # Construct full reward: 12 learned costs + 1 massive hardcoded payout
                full_reward = np.append(self.theta, 100.0) 

                V, policy = self.value_iteration(full_reward)
                expected_svf = self.state_visitation_frequencies(policy)

                # Update gradients only on the 12 behavioral states
                gradient = expert_feat_exp - expected_svf
                self.theta += learning_rate * gradient

            final_reward = np.append(self.theta, 100.0)
            final_V, _ = self.value_iteration(final_reward)
            return final_reward, final_V

    # =========================================================
    # Run Optimization
    # =========================================================
    print("Training Anchored MaxEnt IRL Model...")
    irl_model = MaxEntIRL(trajectories, gamma=0.95)
    inferred_rewards, V_s = irl_model.train(epochs=250, learning_rate=0.05)

    print("\nTraining Complete. Subjective Values V(s) for the 12 states:")
    print("-" * 50)
    for i, val in enumerate(V_s[:12]): # Only print the 12 behavioral states
        s_d = i // 4
        s_g = (i % 4) // 2
        s_n = i % 2
        dist_label = ["Close", "Mid", "Far"][s_d]
        gaze_label = ["Not Looking", "Looking"][s_g]
        noise_label = ["Stable", "Noisy"][s_n]
        print(f"State {i:2d} | Dist: {dist_label:<5} | Gaze: {gaze_label:<11} | Noise: {noise_label:<6} || V(s) = {val:.3f}")

In [ ]:
if 0:
    plot_pull_likelihood(glm_fitting_summary, animal1, fps)
    plot_pull_likelihood(glm_fitting_summary, animal2, fps)
    plot_glm_temporal_filters(glm_fitting_summary, animal1, fps, KERNEL_DURATION_S, N_BASIS_FUNCS)
    plot_glm_temporal_filters(glm_fitting_summary, animal2, fps, KERNEL_DURATION_S, N_BASIS_FUNCS)

In [ ]:
# plot the neural pc1,2,3 traces and some behavioral traces and the bhv glm likelihood

if 0:
    import seaborn as sns
    import pandas as pd
    from scipy.ndimage import gaussian_filter1d  # <--- Imported the smoothing function
    
    likelihood = glm_fitting_summary[(recordedanimal, 'predicted_likelihood')]
    X_all = glm_fitting_summary[(recordedanimal, 'X_all')]
    convolved_vars = glm_fitting_summary[(recordedanimal, 'convolved_var_names')]
    raw_vars = glm_fitting_summary[(recordedanimal, 'raw_var_names')]

    # Align the likelihood/GLM time vector
    abs_time_idx = len(convolved_vars) * N_BASIS_FUNCS + raw_vars.index('abs_time')
    likelihood_time = X_all[:, abs_time_idx] - session_start_time

    # Define behavioral variables for independent panels
    behavior_vars = ['mass_move_speed', 'socialgaze_prob', 'social_evidence']
    behavior_data = data_summary_twoanimals[recordedanimal]
    behavior_time = np.arange(len(behavior_data[0])) / fps - session_start_time
    # ---------------------------------------------------------

    time_point_pull1 = np.array(time_point_pull1)
    time_point_pull2 = np.array(time_point_pull2)

    plot_min_time = 100
    plot_max_time = 450

    time_point_pull1_plot = time_point_pull1[(time_point_pull1 < plot_max_time) & (time_point_pull1 > plot_min_time)]
    time_point_pull2_plot = time_point_pull2[(time_point_pull2 < plot_max_time) & (time_point_pull2 > plot_min_time)]

    ind_FR = (FR_timepoint_allch < plot_max_time) & (FR_timepoint_allch > plot_min_time)
    ind_like = (likelihood_time < plot_max_time) & (likelihood_time > plot_min_time)
    ind_behav = (behavior_time < plot_max_time) & (behavior_time > plot_min_time)

    # --- 2. DYNAMIC FIGURE SETUP ---
    pcs = ['pc1', 'pc2', 'pc3']
    raw_vars_to_plot = ['time_since_pull', 'time_since_succ', 'consec_fails']
    
    # <--- DEFINE SMOOTHING SIGMA HERE
    sigma_smooth = 10
    
    # <--- PRE-CALCULATE SMOOTHED PCs SO PLOTS AND CORRELATIONS MATCH EXACTLY
    smoothed_pcs = {pc: gaussian_filter1d(FRPCs_zscore_allch[pc], sigma=sigma_smooth) for pc in pcs}
    
    # Calculate total panels dynamically
    total_panels = len(pcs) + len(behavior_vars) + 1
    if addpullinfo == 1:
        total_panels += len(raw_vars_to_plot)
    
    fig, axes = plt.subplots(total_panels, 1, figsize=(12, 2.5 * total_panels), sharex=True)

    def draw_pull_lines(ax, y_min, y_max):
        if animal1 == recordedanimal:
            for ipull in time_point_pull1_plot:
                ax.plot([ipull, ipull], [y_min, y_max], 'k')
            for ipull in time_point_pull2_plot:
                ax.plot([ipull, ipull], [y_min, y_max], '--k', alpha=0.6)
        elif animal2 == recordedanimal:
            for ipull in time_point_pull2_plot:
                ax.plot([ipull, ipull], [y_min, y_max], 'k')
            for ipull in time_point_pull1_plot:
                ax.plot([ipull, ipull], [y_min, y_max], '--k', alpha=0.6)

    current_panel = 0

    # --- 3. PLOT NEURAL PCs ---
    for pc in pcs:
        ax = axes[current_panel]
        
        # <--- USE THE SMOOTHED TRACE FOR PLOTTING
        pc_trace = smoothed_pcs[pc][ind_FR]
        
        ax.plot(FR_timepoint_allch[ind_FR], pc_trace, color='tab:blue')
        
        # Safely draw lines handling potential all-NaN slices
        if len(pc_trace) > 0 and not np.all(np.isnan(pc_trace)):
            draw_pull_lines(ax, np.nanmin(pc_trace), np.nanmax(pc_trace))
        
        ax.set_ylabel(f"Neural {pc.upper()}", color='tab:blue')
        current_panel += 1

    # --- 4. PLOT CONTINUOUS BEHAVIORAL VARIABLES ---
    for var_name in behavior_vars:
        ax = axes[current_panel]
        var_idx = data_summary_names.index(var_name)
        var_trace = behavior_data[var_idx][ind_behav]
        
        ax.plot(behavior_time[ind_behav], var_trace, color='tab:orange', linewidth=2)
        
        if len(var_trace) > 0 and not np.all(np.isnan(var_trace)):
            draw_pull_lines(ax, np.nanmin(var_trace), np.nanmax(var_trace))
            
        ax.set_ylabel(var_name, color='tab:orange')
        current_panel += 1

    # --- 5. PLOT GLM RAW HISTORY VARIABLES (CONDITIONAL) ---
    if addpullinfo == 1:
        for r_var in raw_vars_to_plot:
            ax = axes[current_panel]
            
            # Find the exact column index in X_all
            idx = len(convolved_vars) * N_BASIS_FUNCS + raw_vars.index(r_var)
            r_trace = X_all[:, idx][ind_like]
            
            ax.plot(likelihood_time[ind_like], r_trace, color='tab:brown', linewidth=2)
            
            if len(r_trace) > 0 and not np.all(np.isnan(r_trace)):
                draw_pull_lines(ax, np.nanmin(r_trace), np.nanmax(r_trace))
                
            ax.set_ylabel(r_var, color='tab:brown')
            current_panel += 1

    # --- 6. PLOT BEHAVIORAL LIKELIHOOD ---
    ax = axes[current_panel]
    like_trace = likelihood[ind_like]
    
    ax.plot(likelihood_time[ind_like], like_trace, color='purple', linewidth=1.5)
    
    if len(like_trace) > 0 and not np.all(np.isnan(like_trace)):
        draw_pull_lines(ax, np.nanmin(like_trace), np.nanmax(like_trace))
        
    ax.set_ylabel("P(Pull)", color='purple')
    
    # Bottom axis label
    axes[-1].set_xlabel("Time (s)")

    plt.tight_layout()
    plt.show()
    
    
    # =========================================================
    # --- 7. CORRELATION HEATMAP ---
    # =========================================================

    # 1. Create a common time base for the exact window we plotted
    # 1000 points provides high resolution for the interpolation
    common_time = np.linspace(plot_min_time, plot_max_time, 1000)
    corr_dict = {}

    # Helper function to safely interpolate data
    def interpolate_trace(t_original, y_original, t_common):
        # np.interp requires the x-coordinates to be strictly increasing
        idx_sort = np.argsort(t_original)
        return np.interp(t_common, t_original[idx_sort], y_original[idx_sort])

    # 2. Extract and interpolate Neural PCs
    for pc in pcs:
        t_FR = FR_timepoint_allch[ind_FR]
        
        # <--- USE THE SMOOTHED TRACE FOR CORRELATION TOO
        y_FR = smoothed_pcs[pc][ind_FR]
        
        if len(t_FR) > 0:
            corr_dict[f"Neural {pc.upper()}"] = interpolate_trace(t_FR, y_FR, common_time)

    # 3. Extract and interpolate Continuous Behavior
    for var_name in behavior_vars:
        var_idx = data_summary_names.index(var_name)
        t_behav = behavior_time[ind_behav]
        y_behav = behavior_data[var_idx][ind_behav]
        if len(t_behav) > 0:
            corr_dict[var_name] = interpolate_trace(t_behav, y_behav, common_time)

    # 4. Extract and interpolate GLM Raw History Variables
    if addpullinfo == 1:
        for r_var in raw_vars_to_plot:
            idx = len(convolved_vars) * N_BASIS_FUNCS + raw_vars.index(r_var)
            t_like = likelihood_time[ind_like]
            y_raw = X_all[:, idx][ind_like]
            if len(t_like) > 0:
                corr_dict[r_var] = interpolate_trace(t_like, y_raw, common_time)

    # 5. Extract and interpolate GLM Likelihood
    t_like = likelihood_time[ind_like]
    y_like = likelihood[ind_like]
    if len(t_like) > 0:
        corr_dict['P(Pull)'] = interpolate_trace(t_like, y_like, common_time)

    # 6. Compute correlation matrix and plot
    if corr_dict:
        # Convert to DataFrame for easy correlation computation
        df_corr = pd.DataFrame(corr_dict)
        corr_matrix = df_corr.corr()

        # Set up the heatmap figure
        plt.figure(figsize=(10, 8))

        # Draw the heatmap
        sns.heatmap(
            corr_matrix, 
            annot=True,          # Show the correlation values
            cmap='coolwarm',     # Red = Positive, Blue = Negative
            vmin=-1, vmax=1,     # Lock the scale from -1 to 1
            fmt=".2f",           # Round to 2 decimal places
            square=True,         # Make the cells square
            cbar_kws={"shrink": .8} 
        )

        plt.title(f"Cross-Correlation Matrix ({plot_min_time}s - {plot_max_time}s)", pad=20)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print("No valid data found in the specified time window to generate a heatmap.")

In [ ]:
# organize to get the summarizing info about the recording sessions
if 0:
    # ── Derived lists ─────────────────────────────────────────────────────────────
    # partner is whichever animal is NOT the recorded one
    partner_animals = [
        a2 if rec == a1 else a1
        for rec, a1, a2 in zip(recordedanimals, animal1_fixedorders, animal2_fixedorders)
    ]

    # Flatten any array elements to scalars
    totalsessiontime_all_dates = [float(np.squeeze(t)) for t in totalsessiontime_all_dates]
    pull1_num_all_dates        = [float(np.squeeze(p)) for p in pull1_num_all_dates]
    pull2_num_all_dates        = [float(np.squeeze(p)) for p in pull2_num_all_dates]

    # Assign pulls to recorded vs partner
    recorded_pull = [p1 if rec == a1 else p2
                     for rec, a1, p1, p2 in zip(recordedanimals, animal1_fixedorders,
                                                 pull1_num_all_dates, pull2_num_all_dates)]
    partner_pull  = [p2 if rec == a1 else p1
                     for rec, a1, p1, p2 in zip(recordedanimals, animal1_fixedorders,
                                                 pull1_num_all_dates, pull2_num_all_dates)]

    # Count neurons per session from bhvevents_aligned_FR_all_dates
    neuron_counts = []
    for date, rec_animal in zip(dates_list, recordedanimals):
        pull_key = f"{rec_animal} pull"
        n_neurons = len(bhvevents_aligned_FR_all_dates[date][pull_key].keys())
        neuron_counts.append(n_neurons)

    # ── Build dataframe ───────────────────────────────────────────────────────────
    df = pd.DataFrame({
        'session_type':    task_conditions,
        'recorded_animal': recordedanimals,
        'partner_animal':  partner_animals,
        'session_time_s':  totalsessiontime_all_dates,
        'recorded_pull':   recorded_pull,
        'partner_pull':    partner_pull,
        'neuron_count':    neuron_counts,
    })

    # ── Aggregate ─────────────────────────────────────────────────────────────────
    def fmt(mean, mn, mx):
        return f"{mean:.1f} ({int(mn)}-{int(mx)})"

    summary = (df.groupby(['recorded_animal', 'partner_animal', 'session_type'])
                 .agg(
                     num_sessions       = ('session_time_s', 'count'),
                     mean_time_s        = ('session_time_s', 'mean'),
                     min_time_s         = ('session_time_s', 'min'),
                     max_time_s         = ('session_time_s', 'max'),
                     mean_recorded_pull = ('recorded_pull',  'mean'),
                     min_recorded_pull  = ('recorded_pull',  'min'),
                     max_recorded_pull  = ('recorded_pull',  'max'),
                     mean_partner_pull  = ('partner_pull',   'mean'),
                     min_partner_pull   = ('partner_pull',   'min'),
                     max_partner_pull   = ('partner_pull',   'max'),
                     mean_neurons       = ('neuron_count',   'mean'),
                     min_neurons        = ('neuron_count',   'min'),
                     max_neurons        = ('neuron_count',   'max'),
                 )
                 .reset_index())

    summary['session_time_summary']  = summary.apply(
        lambda r: fmt(r['mean_time_s'], r['min_time_s'], r['max_time_s']), axis=1)
    summary['recorded_pull_summary'] = summary.apply(
        lambda r: fmt(r['mean_recorded_pull'], r['min_recorded_pull'], r['max_recorded_pull']), axis=1)
    summary['partner_pull_summary']  = summary.apply(
        lambda r: fmt(r['mean_partner_pull'], r['min_partner_pull'], r['max_partner_pull']), axis=1)
    summary['neuron_summary']        = summary.apply(
        lambda r: fmt(r['mean_neurons'], r['min_neurons'], r['max_neurons']), axis=1)

    summary = summary.drop(columns=[
        'mean_time_s',        'min_time_s',        'max_time_s',
        'mean_recorded_pull', 'min_recorded_pull', 'max_recorded_pull',
        'mean_partner_pull',  'min_partner_pull',  'max_partner_pull',
        'mean_neurons',       'min_neurons',       'max_neurons',
    ])

    print(summary.to_string(index=False))

    # save as csv
    data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+animal1_fixedorders[0]+animal2_fixedorders[0]+'/'
    if not os.path.exists(data_saved_subfolder):
        os.makedirs(data_saved_subfolder)
    #
    summary.to_csv(data_saved_subfolder + "session_summary_with_neuron_number.csv", index=False)
    print("Saved to:", data_saved_subfolder + "session_summary_with_neuron_number.csv")

    #######
    # dataframe with info for each session
    #######
    df_persession = pd.DataFrame({
        'date':            dates_list,
        'session_type':    task_conditions,
        'recorded_animal': recordedanimals,
        'partner_animal':  partner_animals,
        'session_time_s':  totalsessiontime_all_dates,
        'recorded_pull':   recorded_pull,
        'partner_pull':    partner_pull,
        'neuron_count':    neuron_counts,
    }).sort_values(by=['session_type', 'recorded_animal', 'partner_animal']).reset_index(drop=True)


    # print(df_persession.to_string(index=False))

    # Save
    df_persession.to_csv(data_saved_subfolder + "session_persession_with_neuron_number.csv", index=False)
    print("Saved to:", data_saved_subfolder + "session_persession_with_neuron_number.csv")

### plot
#### plot the summarizing FR PCs aligned at the bhv

In [ ]:
if 0:


    bhvevents_aligned_FRPCs_allevents_all_dates_df = pd.DataFrame(columns=['dates','condition','act_animal','bhv_name','clusterID',
                                                           'channelID','FR_allevents'])
    bhvevents_aligned_FRPCs_all_dates_df = pd.DataFrame(columns=['dates','condition','act_animal','bhv_name','clusterID',
                                                           'channelID','FR_average'])

    # reorganize to a dataframes
    for idate in np.arange(0,ndates,1):
        date_tgt = dates_list[idate]
        task_condition = task_conditions[idate]

        bhv_types = list(bhvevents_aligned_FRPCs_allevents_all_dates[date_tgt].keys())

        for ibhv_type in bhv_types:

            clusterIDs = list(bhvevents_aligned_FRPCs_allevents_all_dates[date_tgt][ibhv_type].keys())

            for iclusterID in clusterIDs:

                ichannelID = bhvevents_aligned_FRPCs_allevents_all_dates[date_tgt][ibhv_type][iclusterID]['ch']
                iFR_average = bhvevents_aligned_FRPCs_allevents_all_dates[date_tgt][ibhv_type][iclusterID]['FR_allevents']

                bhvevents_aligned_FRPCs_allevents_all_dates_df = bhvevents_aligned_FRPCs_allevents_all_dates_df.append({'dates': date_tgt, 
                                                                                        'condition':task_condition,
                                                                                        'act_animal':ibhv_type.split()[0],
                                                                                        'bhv_name': ibhv_type.split()[1],
                                                                                        'clusterID':iclusterID,
                                                                                        'channelID':ichannelID,
                                                                                        'FR_allevents':iFR_average,
                                                                                       }, ignore_index=True)

                #
                ichannelID = bhvevents_aligned_FRPCs_all_dates[date_tgt][ibhv_type][iclusterID]['ch']
                iFR_average = bhvevents_aligned_FRPCs_all_dates[date_tgt][ibhv_type][iclusterID]['FR_average']

                bhvevents_aligned_FRPCs_all_dates_df = bhvevents_aligned_FRPCs_all_dates_df.append({'dates': date_tgt, 
                                                                                        'condition':task_condition,
                                                                                        'act_animal':ibhv_type.split()[0],
                                                                                        'bhv_name': ibhv_type.split()[1],
                                                                                        'clusterID':iclusterID,
                                                                                        'channelID':ichannelID,
                                                                                        'FR_average':iFR_average,
                                                                                       }, ignore_index=True)
                
    #
    act_animal_tgt = recordedanimals[0]
    bhv_name_tgt = 'pull'
    condition_tgt = 'MC'
    
    bhvevents_aligned_FRPCs_all_dates_df = bhvevents_aligned_FRPCs_all_dates_df[ \
                                (bhvevents_aligned_FRPCs_all_dates_df['act_animal']==act_animal_tgt) &\
                                (bhvevents_aligned_FRPCs_all_dates_df['bhv_name']==bhv_name_tgt) &\
                                (bhvevents_aligned_FRPCs_all_dates_df['condition']==condition_tgt) ]
    
    
    
    # Create a figure with 3 subplots side-by-side
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    # The three PCs we want to plot
    pcs = ['pc1', 'pc2', 'pc3']

    for i, pc in enumerate(pcs):
        ax = axes[i]

        # Filter the DataFrame for the current PC
        df_pc = bhvevents_aligned_FRPCs_all_dates_df[bhvevents_aligned_FRPCs_all_dates_df['clusterID'] == pc]

        # Loop through each row in the filtered DataFrame to plot the trace
        for index, row in df_pc.iterrows():
            trace = row['FR_average']

            # --- Safety Check for Data Types ---
            # If you loaded this from a CSV, pandas might have read the lists as strings. 
            # If 'trace' is a string, uncomment the line below to convert it back to a list:
            # if isinstance(trace, str): trace = ast.literal_eval(trace)

            # Plot the trace. alpha=0.7 makes overlapping lines slightly transparent
            ax.plot(trace, alpha=0.7, label=str(row['dates']))

        # Formatting each panel
        ax.set_title(f"Trace for {pc.upper()}", fontsize=14)
        ax.set_xlabel("Time Bins", fontsize=12)

        # Only add the Y-axis label to the first (leftmost) plot
        if i == 0:
            ax.set_ylabel("FR Average (PC Projection)", fontsize=12)

    # Add a legend to the last panel to identify the dates
    # bbox_to_anchor moves it slightly outside the plot area so it doesn't cover data
    axes[-1].legend(title="Dates", bbox_to_anchor=(1.05, 1), loc='upper left')

    # Clean up the layout so things don't overlap
    plt.tight_layout()
                
                
                
                